# Music ML — Composer-Conditioned Piano Generation
### Training notebook for Kaggle (GPU)

**Before running:** Enable GPU and internet access in the right-hand sidebar:  
- **Accelerator** → **T4 GPU** or **T4 x2** (do NOT use P100 — PyTorch 2.10+ dropped P100 support)  
- **Internet** → On (needed to install `mido` and `music21`)

Your dataset must be attached. It will appear at `/kaggle/input/music-ml-project/`  
(the exact slug depends on the name you gave it when uploading — check the path in Step 1).

---
### Two modes — pick one:
| Mode | What to run |
|------|-------------|
| **Continue base training** (epochs 1-150+) | Steps 1 → 2 → 2b → 3 → 4 → 5 → 6 (or 6b to resume) |
| **Theory fine-tuning** (Phase 1.5, from epoch 150) | Steps 1 → 2 → 2b → **2c** → 3 → **4b** → **5b** → **6c** |

## Step 1 — Locate the dataset and copy project files to working directory

In [ ]:
import os, shutil, sys

WORK_DIR = '/kaggle/working/music_ml'
CKPT_DIR = '/kaggle/working/checkpoints'
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# ── Find INPUT_DIR: dataset root that contains src/ and scripts/ ──────────────
INPUT_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    depth = root.replace('/kaggle/input', '').count(os.sep)
    if depth > 5:
        dirs[:] = []
        continue
    if os.path.isdir(os.path.join(root, 'src')) and \
       os.path.isdir(os.path.join(root, 'scripts')):
        INPUT_DIR = root
        break

# ── Find PROCESSED_DIR: directory containing composer_map.json ───────────────
# Prefer a path that contains individual composer subdirectories (i.e. real data).
# If multiple datasets have composer_map.json, the one with more subdirs wins.
PROCESSED_DIR = None
best_count = -1
for root, dirs, files in os.walk('/kaggle/input'):
    depth = root.replace('/kaggle/input', '').count(os.sep)
    if depth > 7:
        dirs[:] = []
        continue
    if 'composer_map.json' in files:
        import json
        try:
            with open(os.path.join(root, 'composer_map.json')) as _f:
                _cmap = json.load(_f)
            # Count how many composer subdirs actually exist here
            count = sum(1 for c in _cmap if os.path.isdir(os.path.join(root, c)))
            if count > best_count:
                best_count = count
                PROCESSED_DIR = root
        except Exception:
            pass

if INPUT_DIR is None:
    raise FileNotFoundError(
        'src/ + scripts/ not found under /kaggle/input.\n'
        'Make sure the music-ml project dataset is attached: right sidebar → Add data.'
    )
if PROCESSED_DIR is None:
    raise FileNotFoundError(
        'composer_map.json not found under /kaggle/input.\n'
        'Make sure the music-ml dataset is attached: right sidebar → Add data.'
    )

print(f'Dataset root  : {INPUT_DIR}')
print(f'Processed data: {PROCESSED_DIR}')

# Always re-copy src/ and scripts/ so dataset updates are picked up immediately
for folder in ('src', 'scripts'):
    src = os.path.join(INPUT_DIR, folder)
    dst = os.path.join(WORK_DIR, folder)
    if not os.path.isdir(src):
        raise FileNotFoundError(f'{folder}/ not found in dataset at {src}')
    if os.path.isdir(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f'Copied {folder}/')

sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)
print(f'Working dir   : {os.getcwd()}')

## Step 2 — Install dependencies and verify GPU

In [ ]:
!pip install -q mido music21

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Set Accelerator to T4 GPU in the right sidebar, then restart.')

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
cc_major = torch.cuda.get_device_properties(0).major
cc_minor = torch.cuda.get_device_properties(0).minor
sm       = f'sm_{cc_major}{cc_minor}'

print(f'GPU  : {gpu_name}')
print(f'VRAM : {vram_gb} GB')
print(f'CUDA capability: {sm}')

if cc_major < 7:
    raise RuntimeError(
        f'GPU {gpu_name} ({sm}) is not supported by PyTorch 2.0+.\n'
        'Switch to a T4 GPU: right sidebar → Accelerator → T4 GPU → Save → re-run.'
    )

print('\nGPU is compatible. Ready to proceed.')

## Step 2b — Process GiantMIDI dataset and merge with MAESTRO

GiantMIDI is auto-detected from `/kaggle/input/`. If not attached, this step is skipped and training continues on MAESTRO only.

**To attach:** right sidebar → **Add data** → search **giantmidi** → add the dataset.

Only the composers already present in `composer_map.json` are kept; all others are ignored.

In [ ]:
import json, re, shutil
from pathlib import Path
import numpy as np

# Set this to your GiantMIDI path if auto-detection fails, otherwise leave as None
GIANTMIDI_DIR = '/kaggle/input/datasets/pictureinthenoise/music-generation-with-giantmidi-piano/giantmidi-piano-unzipped-midi-v1.21-clean'

# ── 1. Create combined processed dir and symlink MAESTRO .npy files ───────────
COMBINED_DIR = '/kaggle/working/processed_combined'
os.makedirs(COMBINED_DIR, exist_ok=True)

shutil.copy2(
    os.path.join(PROCESSED_DIR, 'composer_map.json'),
    os.path.join(COMBINED_DIR,  'composer_map.json'),
)

with open(os.path.join(COMBINED_DIR, 'composer_map.json')) as _f:
    _cmap = json.load(_f)

for _composer in _cmap:
    _src_dir = os.path.join(PROCESSED_DIR, _composer)
    _dst_dir = os.path.join(COMBINED_DIR,  _composer)
    if not os.path.isdir(_src_dir):
        continue
    os.makedirs(_dst_dir, exist_ok=True)
    for _fname in os.listdir(_src_dir):
        if _fname.endswith('.npy'):
            _lnk = os.path.join(_dst_dir, _fname)
            if not os.path.exists(_lnk):
                os.symlink(os.path.join(_src_dir, _fname), _lnk)

print(f'MAESTRO data linked into {COMBINED_DIR}')

# ── 2. Collect GiantMIDI files ────────────────────────────────────────────────
_giant_mids = []
if GIANTMIDI_DIR and os.path.isdir(GIANTMIDI_DIR):
    for _root, _, _files in os.walk(GIANTMIDI_DIR):
        for _f in _files:
            if _f.lower().endswith(('.mid', '.midi')):
                _giant_mids.append(os.path.join(_root, _f))
else:
    for _root, _dirs, _files in os.walk('/kaggle/input'):
        _depth = _root.replace('/kaggle/input', '').count(os.sep)
        if _depth > 5:
            _dirs[:] = []
            continue
        for _f in _files:
            if _f.lower().endswith(('.mid', '.midi')):
                _giant_mids.append(os.path.join(_root, _f))

if not _giant_mids:
    print('GiantMIDI not detected — training on MAESTRO only.')
    print('Attach via: right sidebar → Add data → search giantmidi')
else:
    print(f'GiantMIDI: {len(_giant_mids)} MIDI files found.')

    def _composer_to_slug(name):
        parts = name.lower().split()
        return '-'.join([parts[-1]] + parts[:-1]) if len(parts) > 1 else parts[0]

    _cmap_keys = list(_cmap.keys())
    _slug_to_composer = {}
    for _k in _cmap_keys:
        _slug = _composer_to_slug(_k)
        _slug_to_composer[_slug] = _k
        _parts = _slug.split('-')
        if len(_parts) > 2:
            _slug_to_composer['-'.join(_parts[:2])] = _k

    def _match_filename(filepath):
        stem = os.path.splitext(os.path.basename(filepath))[0]
        for _slug in sorted(_slug_to_composer, key=len, reverse=True):
            if stem.startswith(_slug + '-'):
                return _slug_to_composer[_slug]
        return None

    _gm_groups = {}
    _unmatched  = set()
    for _mp in _giant_mids:
        _matched = _match_filename(_mp)
        if _matched:
            _gm_groups.setdefault(_matched, []).append(_mp)
        else:
            _unmatched.add(os.path.basename(_mp).split('-')[0])

    _total_gm = sum(len(v) for v in _gm_groups.values())
    print(f'Matched {_total_gm} GiantMIDI files across {len(_gm_groups)} composers.')
    if _unmatched:
        print(f'Skipped surnames not in vocabulary: {sorted(_unmatched)[:15]} ...')
    for _c, _fps in sorted(_gm_groups.items()):
        print(f'  {_c}: {len(_fps)} GiantMIDI files')

    from src.data.midi_parser import midi_to_events

    _proc, _errs = 0, 0
    for _composer, _paths in _gm_groups.items():
        _out_dir = os.path.join(COMBINED_DIR, _composer)
        os.makedirs(_out_dir, exist_ok=True)
        for _mp in _paths:
            _stem    = 'gm_' + Path(_mp).stem
            _out_npy = os.path.join(_out_dir, _stem + '.npy')
            if os.path.exists(_out_npy):
                continue
            try:
                _tokens = midi_to_events(_mp)
                if len(_tokens) < 20:
                    continue
                np.save(_out_npy, np.array(_tokens, dtype=np.int64))
                _proc += 1
            except Exception as _e:
                _errs += 1

    print(f'\nProcessed {_proc} GiantMIDI files ({_errs} parse errors).')

PROCESSED_DIR = COMBINED_DIR
print(f'\nPROCESSED_DIR → {PROCESSED_DIR}')

## Step 2c — Extract theory labels  *(Theory fine-tuning only — skip for base training)*

This cell analyses every MIDI file with **music21** and writes a parallel
`piece_theory.npy` file (shape `[N, 4]`, columns: key / chord / beat / cadence)
alongside each existing `piece.npy` token file.

**Runtime:** ~2–4 hours for the full MAESTRO+GiantMIDI set on a T4.  
Run it once; subsequent sessions detect existing `_theory.npy` files and skip them.

**If you have already extracted and re-uploaded theory labels as a Kaggle dataset,**
attach that dataset, set `THEORY_LABELS_DIR` to its path, and run the symlink
block at the bottom of this cell instead of the extraction block.

In [ ]:
# ── Option A: extract theory labels from scratch ──────────────────────────────
# Set RUN_EXTRACTION = False if you are loading pre-computed labels (Option B).
RUN_EXTRACTION = True

# ── Option B: symlink pre-computed theory labels ──────────────────────────────
# After extracting once, download /kaggle/working/processed_combined and
# re-upload as a Kaggle dataset.  Then set the path below and RUN_EXTRACTION=False.
THEORY_LABELS_DIR = None   # e.g. '/kaggle/input/music-ml-theory-labels/processed_combined'

import glob
from pathlib import Path
from src.data.theory_extractor import extract_theory_labels, tokens_to_timestamps_ms

with open(os.path.join(PROCESSED_DIR, 'composer_map.json')) as _f:
    _cmap_theory = json.load(_f)

if not RUN_EXTRACTION and THEORY_LABELS_DIR and os.path.isdir(THEORY_LABELS_DIR):
    # ── Option B: symlink _theory.npy files from pre-computed dataset ─────────
    linked = 0
    for composer in _cmap_theory:
        src_dir = os.path.join(THEORY_LABELS_DIR, composer)
        dst_dir = os.path.join(PROCESSED_DIR,     composer)
        if not os.path.isdir(src_dir):
            continue
        os.makedirs(dst_dir, exist_ok=True)
        for fname in os.listdir(src_dir):
            if fname.endswith('_theory.npy'):
                lnk = os.path.join(dst_dir, fname)
                if not os.path.exists(lnk):
                    os.symlink(os.path.join(src_dir, fname), lnk)
                    linked += 1
    print(f'Symlinked {linked} pre-computed theory files.')

elif RUN_EXTRACTION:
    # ── Option A: extract from source MIDIs ───────────────────────────────────
    # Build a lookup: npy_stem → midi_path by scanning all available MIDI files
    print('Building MIDI path index (scanning /kaggle/input)...')
    _midi_index = {}   # stem (without extension) → full path
    for _root, _dirs, _files in os.walk('/kaggle/input'):
        _depth = _root.replace('/kaggle/input', '').count(os.sep)
        if _depth > 8:
            _dirs[:] = []
            continue
        for _f in _files:
            if _f.lower().endswith(('.mid', '.midi')):
                _stem = Path(_f).stem
                _midi_index[_stem] = os.path.join(_root, _f)
    print(f'Index built: {len(_midi_index)} MIDI files.')

    def _find_midi(npy_path: str):
        """Return source MIDI path for a given .npy path, or None."""
        stem = Path(npy_path).stem
        # GiantMIDI files are stored as 'gm_<original_stem>.npy'
        if stem.startswith('gm_'):
            stem = stem[3:]
        return _midi_index.get(stem)

    processed_count = 0
    skipped_count   = 0
    error_count     = 0

    for composer in sorted(_cmap_theory.keys()):
        comp_dir = os.path.join(PROCESSED_DIR, composer)
        if not os.path.isdir(comp_dir):
            continue
        npy_files = sorted(
            f for f in os.listdir(comp_dir)
            if f.endswith('.npy') and not f.endswith('_theory.npy')
        )
        for npy_fname in npy_files:
            npy_path    = os.path.join(comp_dir, npy_fname)
            theory_path = npy_path.replace('.npy', '_theory.npy')

            if os.path.exists(theory_path):
                skipped_count += 1
                continue

            midi_path = _find_midi(npy_path)
            if midi_path is None:
                skipped_count += 1
                continue

            try:
                tokens     = np.load(npy_path)
                timestamps = tokens_to_timestamps_ms(tokens)
                labels     = extract_theory_labels(midi_path, tokens, timestamps)
                np.save(theory_path, labels)
                processed_count += 1
                if processed_count % 50 == 0:
                    print(f'  [{composer}] processed {processed_count} files...')
            except Exception as e:
                error_count += 1
                # Don't crash on a single file failure

    print(f'\nTheory extraction complete.')
    print(f'  Extracted : {processed_count}')
    print(f'  Skipped   : {skipped_count}  (already done or no MIDI found)')
    print(f'  Errors    : {error_count}')
else:
    print('Skipping theory extraction. Set RUN_EXTRACTION=True to extract.')
    print('Pieces without _theory.npy will use all-unknown labels (safe — just no theory supervision).')

## Step 3 — Verify processed data

In [ ]:
import json

with open(os.path.join(PROCESSED_DIR, 'composer_map.json')) as f:
    composer_map = json.load(f)

total_files  = 0
theory_files = 0
for composer in composer_map:
    d = os.path.join(PROCESSED_DIR, composer)
    if os.path.isdir(d):
        fnames = os.listdir(d)
        n      = sum(1 for x in fnames if x.endswith('.npy') and not x.endswith('_theory.npy'))
        t      = sum(1 for x in fnames if x.endswith('_theory.npy'))
        total_files  += n
        theory_files += t

print(f'Composers    : {len(composer_map)}')
print(f'Token files  : {total_files}')
print(f'Theory files : {theory_files}  ({100*theory_files//max(1,total_files)}% coverage)')
print()
for name, idx in sorted(composer_map.items(), key=lambda x: x[1]):
    d = os.path.join(PROCESSED_DIR, name)
    n = len(os.listdir(d)) if os.path.isdir(d) else 0
    print(f'  {idx:2d}: {name} ({n} files)')

## Step 4 — Configure training  *(base training — skip to Step 4b for theory fine-tuning)*

In [ ]:
from src.training.config import TrainConfig

cfg = TrainConfig(
    processed_data_dir = PROCESSED_DIR,
    checkpoint_dir     = CKPT_DIR,

    # Model
    d_model            = 512,
    n_heads            = 8,
    n_layers           = 6,
    d_ff               = 2048,
    dropout            = 0.1,
    num_composers      = len(composer_map),
    composer_embed_dim = 64,

    # Training
    seq_len            = 512,
    stride             = 256,
    batch_size         = 32,
    num_epochs         = 150,
    learning_rate      = 5e-5,
    warmup_steps       = 4000,
    save_every         = 5,
    log_every          = 200,

    # Theory disabled for base training
    use_theory         = False,
)

print('Base training config ready. Checkpoint dir:', CKPT_DIR)

## Step 4b — Configure theory fine-tuning  *(Phase 1.5 — replaces Step 4)*

Run this cell **instead of Step 4** when fine-tuning from the epoch-150 backbone.

In [ ]:
from src.training.config import TrainConfig

cfg = TrainConfig(
    processed_data_dir = PROCESSED_DIR,
    checkpoint_dir     = CKPT_DIR,

    # ── Architecture — MUST match the backbone checkpoint exactly ─────────────
    d_model            = 512,
    n_heads            = 8,
    n_layers           = 6,
    d_ff               = 2048,
    dropout            = 0.1,
    num_composers      = len(composer_map),
    composer_embed_dim = 64,

    # ── Fine-tuning schedule ──────────────────────────────────────────────────
    seq_len            = 512,
    stride             = 256,
    batch_size         = 16,       # halved — theory labels add ~20% overhead
    num_epochs         = 30,       # short fine-tune on top of 150 base epochs
    learning_rate      = 5e-6,    # 10× lower than base LR
    warmup_steps       = 500,
    weight_decay       = 1e-2,
    grad_clip          = 1.0,
    save_every         = 5,
    log_every          = 200,

    # ── Theory flags ─────────────────────────────────────────────────────────
    use_theory         = True,     # enable key/chord/beat conditioning + aux losses
    theory_loss_weight = 0.1,      # weight for chord/key auxiliary losses
    freeze_base_embed  = True,     # freeze token embed rows 0-391 (pre-trained vocab)
    use_era            = True,     # add era embedding (Baroque/Classical/Romantic/Impressionist/Modern)
)

print('Theory fine-tuning config ready.')
print(f'  LR             : {cfg.learning_rate}  (10x lower than base)')
print(f'  Epochs         : {cfg.num_epochs}')
print(f'  Theory loss w  : {cfg.theory_loss_weight}')
print(f'  Freeze base emb: {cfg.freeze_base_embed}')
print(f'  Use era embed  : {cfg.use_era}')


## Step 5 — Build dataset and model  *(base training)*

In [ ]:
from src.training.trainer import Trainer

trainer = Trainer(cfg, composer_map)
print('Ready to train.')

## Step 5b — Load backbone checkpoint for theory fine-tuning  *(replaces Step 5)*

Attach your checkpoint dataset (right sidebar → Add data), then run this cell.
It builds the theory-extended model, loads the epoch-150 backbone weights
(strict=False so new theory layers stay randomly initialised), and optionally
freezes the pre-trained token embedding rows.

In [ ]:
import torch, glob
from src.training.trainer import Trainer

# ── Point this at whichever Kaggle dataset holds your epoch-150 checkpoint ───
CKPT_INPUT_DIR = '/kaggle/input/datasets/wpyggg/music-ml-checkpoints/music_ml_checkpoints'

# Pick the latest epoch checkpoint, falling back to checkpoint_best.pt
epoch_ckpts   = sorted(glob.glob(os.path.join(CKPT_INPUT_DIR, 'checkpoint_epoch*.pt')))
backbone_path = epoch_ckpts[-1] if epoch_ckpts else \
                os.path.join(CKPT_INPUT_DIR, 'checkpoint_best.pt')
print(f'Backbone checkpoint : {backbone_path}')

# Build the theory-extended trainer (use_theory=True in cfg)
trainer = Trainer(cfg, composer_map)

ckpt  = torch.load(backbone_path, map_location='cpu', weights_only=False)
raw   = trainer._raw_model()
state = ckpt['model_state']

# ── Handle composer_embed size mismatch ───────────────────────────────────────
# The checkpoint may have been saved with fewer composers (e.g. 43) than the
# current model (68).  PyTorch strict=False skips missing/extra keys but still
# raises on shape mismatches, so we transplant the rows manually first.
ckpt_ce = state.get('composer_embed.weight')
curr_ce = raw.composer_embed.weight
if ckpt_ce is not None and ckpt_ce.shape != curr_ce.shape:
    n_old = ckpt_ce.shape[0]
    n_new = curr_ce.shape[0]
    print(f'composer_embed size mismatch: checkpoint={list(ckpt_ce.shape)}, '
          f'model={list(curr_ce.shape)}')
    print(f'Transplanting {n_old} trained rows; {n_new - n_old} new rows stay random.')
    with torch.no_grad():
        curr_ce[:n_old] = ckpt_ce
    del state['composer_embed.weight']   # remove so load_state_dict won't re-check it

missing, unexpected = raw.load_state_dict(state, strict=False)

print(f'\nMissing keys  (new theory/era layers — expected): {len(missing)}')
for k in missing[:15]:
    print(f'  {k}')
print(f'Unexpected keys (should be 0)                   : {len(unexpected)}')

backbone_epoch = ckpt.get('epoch', '?')
backbone_val   = ckpt.get('val_loss', float('inf'))
print(f'\nBackbone loaded from epoch {backbone_epoch}  (val loss {backbone_val:.4f})')

# Freeze old token embedding rows (0-391) to protect pre-trained knowledge
trainer.freeze_old_embeddings()

print('\nReady for theory fine-tuning.')

## Step 6 — Train  *(base training — fresh start)*

> Checkpoints are saved to `/kaggle/working/checkpoints/` every 5 epochs and on every validation loss improvement.  
> Kaggle sessions run for up to **12 hours** even with the browser closed.  
> After the session ends, download checkpoints from the **Output** tab on the right, or see Step 6b to resume.

In [ ]:
# trainer.train()

## Step 6b — Resume base training from a previous session

1. After a session ends, go to your notebook page → **Output** tab → download the checkpoints zip.
2. Upload that zip as a **new Kaggle dataset** (e.g. `music-ml-checkpoints`).
3. Attach it to this notebook in the right sidebar → **Add data**.
4. Run Steps 1–5, then run this cell instead of Step 6.

In [ ]:
import torch, glob

CKPT_INPUT_DIR = '/kaggle/input/datasets/wpyggg/music-ml-checkpoints/music_ml_checkpoints'

epoch_ckpts = sorted(glob.glob(os.path.join(CKPT_INPUT_DIR, 'checkpoint_epoch*.pt')))
resume_path = epoch_ckpts[-1] if epoch_ckpts else os.path.join(CKPT_INPUT_DIR, 'checkpoint_best.pt')
print(f'Resuming from: {resume_path}')

from src.training.trainer import Trainer

ckpt = torch.load(resume_path, map_location='cpu', weights_only=False)
cfg_saved          = ckpt['config']
composer_map_saved = ckpt['composer_map']
start_epoch        = ckpt['epoch'] + 1
best_val           = ckpt['val_loss']

print(f'Resuming from epoch {start_epoch} (best val loss so far: {best_val:.4f})')

cfg_saved.processed_data_dir = PROCESSED_DIR
cfg_saved.checkpoint_dir     = CKPT_DIR

trainer = Trainer(cfg_saved, composer_map_saved)
trainer._raw_model().load_state_dict(ckpt['model_state'])
trainer.optimizer.load_state_dict(ckpt['optim_state'])

trainer.train(start_epoch=start_epoch, best_val=best_val)

## Step 6c — Theory fine-tuning  *(run after Steps 4b and 5b)*

Runs 30 fine-tuning epochs from the backbone checkpoint.
The model learns key/chord/beat conditioning while preserving all note-level
statistics from the 150-epoch base training.

Checkpoints are saved to `/kaggle/working/checkpoints/` with the prefix
`checkpoint_theory_*` so they do not overwrite your base training checkpoints.

## Step 6d — Resume theory fine-tuning from a previous session  *(optional)*

Use this instead of Steps 5b + 6c when you already have `checkpoint_theory_epoch*.pt` files
from a previous run and want to continue (e.g. extend from 30 → 100 epochs).

1. Upload the theory checkpoint dataset to Kaggle (or reuse the output from the previous run).
2. Set `THEORY_CKPT_INPUT_DIR` and `EXTRA_EPOCHS` below, then run this cell.

In [ ]:
import torch, glob, types
from src.training.config import TrainConfig
from src.training.trainer import Trainer

# ── Configuration ─────────────────────────────────────────────────────────────
THEORY_CKPT_INPUT_DIR = '/kaggle/input/datasets/wpyggg/music-ml-checkpoints/music_ml_checkpoints'
EXTRA_EPOCHS          = 70   # how many MORE epochs to run (e.g. 70 to go from 30 → 100)

# ── Find the latest theory checkpoint ────────────────────────────────────────
epoch_ckpts   = sorted(glob.glob(os.path.join(THEORY_CKPT_INPUT_DIR, 'checkpoint_theory_epoch*.pt')))
resume_path   = epoch_ckpts[-1] if epoch_ckpts else \
                os.path.join(THEORY_CKPT_INPUT_DIR, 'checkpoint_theory_best.pt')
print(f'Resuming theory fine-tuning from: {resume_path}')

ckpt = torch.load(resume_path, map_location='cpu', weights_only=False)

# Restore config from checkpoint and extend epoch count
cfg_r = ckpt['config']
cfg_r.processed_data_dir = PROCESSED_DIR
cfg_r.checkpoint_dir     = CKPT_DIR
cfg_r.num_epochs         = ckpt['epoch'] + EXTRA_EPOCHS   # absolute target epoch

start_epoch = ckpt['epoch'] + 1
best_val    = ckpt['val_loss']
print(f'Resuming from epoch {ckpt["epoch"]}  (val loss {best_val:.4f})')
print(f'Training to epoch  {cfg_r.num_epochs}  ({EXTRA_EPOCHS} more epochs)')

# ── Rebuild trainer and restore full state ────────────────────────────────────
composer_map_r = ckpt['composer_map']
trainer = Trainer(cfg_r, composer_map_r)
trainer._raw_model().load_state_dict(ckpt['model_state'])
trainer.optimizer.load_state_dict(ckpt['optim_state'])
trainer.scheduler.load_state_dict(ckpt['scheduler_state'])

# ── Override checkpoint naming (same as Step 6c) ──────────────────────────────
def _save_theory_checkpoint(self, epoch, val_loss, tag):
    path = os.path.join(self.cfg.checkpoint_dir, f'checkpoint_theory_{tag}.pt')
    torch.save({
        'epoch':           epoch,
        'val_loss':        val_loss,
        'model_state':     self._raw_model().state_dict(),
        'optim_state':     self.optimizer.state_dict(),
        'scheduler_state': self.scheduler.state_dict(),
        'config':          self.cfg,
        'composer_map':    self.composer_map,
    }, path)
    print(f'  Saved: {path}')

trainer._save_checkpoint = types.MethodType(_save_theory_checkpoint, trainer)

print('\nStarting resumed theory fine-tuning...')
trainer.train(start_epoch=start_epoch, best_val=best_val)

In [ ]:
# Override checkpoint naming so theory checkpoints are clearly labelled
import types

def _save_theory_checkpoint(self, epoch, val_loss, tag):
    import torch, os
    path = os.path.join(self.cfg.checkpoint_dir, f'checkpoint_theory_{tag}.pt')
    torch.save({
        'epoch':           epoch,
        'val_loss':        val_loss,
        'model_state':     self._raw_model().state_dict(),
        'optim_state':     self.optimizer.state_dict(),
        'scheduler_state': self.scheduler.state_dict(),
        'config':          self.cfg,
        'composer_map':    self.composer_map,
    }, path)
    print(f'  Saved: {path}')

trainer._save_checkpoint = types.MethodType(_save_theory_checkpoint, trainer)

print('Starting theory fine-tuning...')
print(f'  Epochs        : {cfg.num_epochs}')
print(f'  LR            : {cfg.learning_rate}')
print(f'  Theory weight : {cfg.theory_loss_weight}')
print(f'  Backbone val  : {backbone_val:.4f}  (for reference only)')
print()

# Reset best_val to inf so "theory_best" tracks the best epoch *within* fine-tuning,
# not whether we beat the 150-epoch base model on raw token loss.
trainer.train(start_epoch=1, best_val=float('inf'))

## Step 7 — Generate a sample MIDI

In [ ]:
import torch
from src.inference.generate import load_model_from_checkpoint, generate
from src.data.midi_parser import events_to_midi

COMPOSER = 'Frédéric Chopin'   # change to any composer name from the map above
OUTPUT   = '/kaggle/working/generated.mid'

# Use the theory checkpoint if it exists, otherwise fall back to the best base checkpoint
theory_best = os.path.join(CKPT_DIR, 'checkpoint_theory_best.pt')
base_best   = os.path.join(CKPT_DIR, 'checkpoint_best.pt')
gen_ckpt    = theory_best if os.path.exists(theory_best) else base_best
print(f'Generating from: {gen_ckpt}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, cmap, _ = load_model_from_checkpoint(gen_ckpt, device)

MAX_RETRIES = 3

if COMPOSER not in cmap:
    print(f'Unknown composer. Available: {list(cmap.keys())}')
else:
    for attempt in range(1, MAX_RETRIES + 1):
        attempt_temp = 1.0 * (1.0 + 0.3 * (attempt - 1))  # 1.0 → 1.3 → 1.6
        tokens = generate(
            model         = model,
            composer_id   = cmap[COMPOSER],
            device        = device,
            max_tokens    = 4096,
            temperature   = attempt_temp,
            top_k         = 50,
            top_p         = 0.95,
            min_tokens    = 1024,
            composer_name = COMPOSER,
        )
        note_count = sum(1 for t in tokens if 0 <= t <= 127)
        print(f'Attempt {attempt} (T={attempt_temp:.1f}): {len(tokens)} tokens, {note_count} NOTE_ON events')
        if note_count >= 10:
            break
        if attempt < MAX_RETRIES:
            print('  Too few notes — retrying with higher temperature...')
    if note_count < 10:
        print('Warning: very few notes after retries. Try USE_THEORY_CHECKPOINT=False ')
        print('to check if the base model works, or re-run theory fine-tuning.')
    events_to_midi(tokens, OUTPUT)
    print(f'Generated {len(tokens)} tokens → {OUTPUT}')
    print('Download it from the Output tab on the right sidebar.')
